Groundwater | Flow Modeling Track

# Step 1: Model Goal – Defining the Flow Model Problem

> **The 10 steps at a glance:** **1-Goal** → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**The journey:** Before building any model, we must define *what* we're trying to achieve. This first step establishes clear objectives that guide every subsequent decision.
| **Core content** | ~10 minutes |
|:--|:--|
| **Optional: Exercises & real-world context** | +5 minutes |

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** clear objectives for a groundwater flow model
2. **Identify** the appropriate spatial and temporal scales for a modeling problem
3. **List** the essential data requirements for flow modeling
4. **Recognize** key stakeholders and their model requirements

---

## Prerequisites

Before starting this notebook, you should have:
- **Completed [0_start_here.ipynb](../0_start_here.ipynb)** – this explains the 10-step modeling framework that structures the entire course
- Basic understanding of hydrogeology concepts (Darcy's Law, aquifer types)
- Familiarity with the Limmat Valley case study context


---

## Introduction

Before we dive into coding or running simulations, every robust groundwater model starts with a **sharp definition of the problem**.

In this notebook, we define the goals and scope of our **groundwater flow model** for the Limmat Valley aquifer. We'll answer:
- **What** should the model simulate?
- **Where** and at what scale?
- **What data** do we need?
- **Who** will use the results?

In [ ]:
# Setup for displaying study area map
import sys
sys.path.append('../../_SUPPORT/src')

from data_utils import download_named_file
from map_utils import plot_model_area_map

### Study Area Overview

The Limmat Valley aquifer is located in and around Zürich, Switzerland. The map below shows the aquifer extent (colored by thickness) and the model boundary we'll use for our simulations.

In [ ]:
# Download and display study area map
gw_map_path = download_named_file(name='groundwater_map_norm', data_type='gis')
model_boundary_path = download_named_file(name='model_boundary', data_type='gis')

plot_model_area_map(
    gw_depth_path=gw_map_path,
    model_boundary_path=model_boundary_path,
    custom_title="Limmat Valley study area with model boundary (black outline)",
    basemap="osm",
    basemap_alpha=0.8
)

---

## 🎯 Model Objectives – What Are We Solving?

What exactly do we want our model to answer? Clearly defined objectives ensure your model serves a purpose and remains scientifically defensible.

### Primary Objectives

Our Limmat Valley flow model should be able to:

1. **Simulate steady-state groundwater flow** through the alluvial aquifer
2. **Reproduce observed hydraulic heads** at monitoring wells
3. **Quantify groundwater-surface water exchange** with the Limmat and Sihl rivers
4. **Evaluate the impact of pumping** on the aquifer system
5. **Support scenario analysis** for water management decisions

### Educational Objectives

As a teaching tool, the model should also:

- Demonstrate key concepts in groundwater flow modeling
- Be simple enough to understand yet realistic enough to be meaningful
- Run quickly enough for interactive exploration
- Be based on **open-source** software compatible with the JupyterHub environment
- Answer a variety of case study questions on common flow problems
- Serve as a foundation for transport modeling

<details>
<summary><strong>Optional: Exercise – Refining Model Objectives</strong></summary>

> ✏️ **Exercise: Refining Model Objectives**
> 
> Consider the objectives listed above. For a real-world consulting project:
> 
> 1. Which objectives would be most important for a water utility managing the aquifer?
> 2. What additional objectives might a regulator require?
> 3. How might objectives differ if contamination was a concern?
>
> **Hint:** Think about what each stakeholder is legally responsible for and what decisions they need to make.

</details>

---

## 🗺️ Setting the Scene – Where and When?

Models don't exist in a vacuum. Choose a meaningful spatial and temporal scale.

### Spatial Scale

| Dimension | Value | Rationale |
|-----------|-------|----------|
| **Domain extent** | ~15 km along valley | Covers the main aquifer from Zürich to downstream |
| **Width** | 1-3 km | Follows valley boundaries |
| **Depth** | ~50 m | Captures the main productive aquifer |
| **Grid resolution** | 50-100 m | Balance between detail and computation |

### Temporal Scale

| Aspect | Choice | Rationale |
|--------|--------|----------|
| **Initial focus** | Steady-state | Understand average conditions first |
| **Future extension** | Transient (daily-monthly) | For seasonal and event-based analysis |
| **Calibration period** | Multi-year average | Robust parameter estimation |

---

## Essential Data – What Do We Need?

Start compiling your data wishlist early. Check both availability and quality before committing to complex scenarios.

### Required Data Categories

| Category | Data Type | Source | Availability |
|----------|-----------|--------|-------------|
| **Geometry** | Aquifer boundaries, thickness | Geological surveys | Good |
| **Topography** | Surface DEM (digital elevation model), aquifer base | SwissTopo, boreholes | Good |
| **Hydraulic properties** | *K* (hydraulic conductivity – how easily water flows through rock), *Ss* (specific storage), *Sy* (specific yield) | AWEL (Amt für Abfall, Wasser, Energie und Luft – Cantonal Office for Waste, Water, Energy and Air), literature | Sparse |
| **Boundary conditions** | River stages, lake levels | BAFU (Bundesamt für Umwelt – Federal Office for the Environment) gauging stations | Good |
| **Stresses** | Pumping rates, recharge | Water utilities, climate data | Good |
| **Observations** | Groundwater levels | AWEL monitoring network | Good |


> **Key terms:** 
> - *K* (hydraulic conductivity): How easily water flows through the aquifer material (m/day)
> - *Ss* (specific storage): Water released from storage per unit head drop in confined aquifers (1/m)
> - *Sy* (specific yield): Drainable porosity in unconfined aquifers (dimensionless, typically 0.1–0.3)
> - *DEM*: Digital Elevation Model – a grid of surface elevations

### Data Quality Considerations

- **Temporal coverage**: Ideally 10+ years for robust calibration
- **Spatial coverage**: Are there data gaps in critical areas?
- **Consistency**: Different sources may use different datums or formats
- **Uncertainty**: What are the measurement errors?

### Comprehension Check

Before moving on, make sure you can answer this question:

> **Which data category in the table above is marked as "Sparse" and will likely need to be estimated through model calibration?**

<details>
<summary>Click to check your answer</summary>

**Hydraulic properties** (K, Ss, Sy) – These are difficult to measure directly across the entire aquifer. We typically have only a few pumping test measurements. Model calibration adjusts these values until simulated groundwater levels match observations.

This is why Step 5 (Calibration) exists in the modeling process!
</details>

---

## 👥 Key Players – Who Cares About the Results?

A model's value depends on its relevance. Understanding your end users should guide your design decisions.

### Primary Users

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Students** | Learning modeling concepts | Clear, well-documented, fast |
| **Instructors** | Teaching demonstrations | Flexible, illustrative results |

<details>
<summary><strong>Optional: Real-world stakeholders</strong></summary>

### In a Real-World Context

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Water utilities** | Sustainable abstraction | Drawdown predictions, safe yield |
| **Regulators** | Resource protection | Compliance scenarios, uncertainty |
| **Planners** | Development impacts | What-if scenarios, cumulative effects |
| **Public** | Water security | Clear communication of results |

</details>

---

## Summary: Flow Model Goal Definition

> **🎯 Model Objectives**
> 
> Create an educational groundwater flow model of the Limmat Valley aquifer that:
> - Simulates steady-state (and later transient) groundwater flow
> - Reproduces observed hydraulic heads
> - Quantifies groundwater-surface water interaction
> - Supports scenario analysis and teaching
>
> **🗺️ Setting the Scene**
>
> - Spatial: ~15 km valley length, 50-100 m grid resolution
> - Temporal: Steady-state initially, transient extension planned
>
> **📊 Essential Data**
>
> - Geometry and topography: Well-constrained
> - Hydraulic properties: Sparse, will require calibration
> - Boundary conditions and observations: Good coverage
>
> **👥 Key Players**
>
> - Primary: Students and instructors (educational use)
> - Secondary: Demonstrates workflow applicable to real-world users

---

## Next Steps

With our model goals clearly defined, we're ready to move on to **Step 2: The Perceptual Model**, where we'll build a detailed understanding of the Limmat Valley aquifer system — its geometry, water balance, and key processes.

**Continue to:** [2_perceptual_model.ipynb](2_perceptual_model.ipynb)

---

# Step 1: Exercises - Darcy's law 

## Part 1 - Derivation

Darcy's law is essential when it comes to flow in an aquifer. 
Let us look at a standard Darcy experiement set-up to refresh our knowledge about Darcy's law and prepare for the next notebooks.

We want to derive experimentally Darcy's law, which relates 2 measurable quantities:
- **Hydraulic gradient** $I$ = $\frac{\Delta h}{L}$, with $\Delta h$ the hydraulic gradient (ie. hydraulic head differenve between the input and the output of the column)
- **Specific discharge** $q$ at the input (or output, since a steady state flow is assumed).

You might later need some additionnal parameters regarding the experiment setup:
- Column radius $r = d/2 = 0.127$ mm
- Column length $L = 1$ m

The experiment below repeats $q$ measurements for different $\Delta h$ values, to plot $q$ against $I$.

In [ ]:
# Import local libraries
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')
from shared_functions import check_task_with_solution
import print_images as du
from darcy_law_experiment import darcy_task_1_1, darcy_task_1_2, darcy_task_1_3
from darcy_several_layers_keq import exercise_aquifers_layered_keq_attribution

du.display_image(
    image_filename='DarcyExperimentSetup.png', 
    image_folder='3_exercises', 
    )

darcy_task_1_1()
darcy_task_1_2()
darcy_task_1_3()

check_task_with_solution("task03_1")

## Part 2 - Use for aquifer systems

In [ ]:
exercise_aquifers_layered_keq_attribution()
check_task_with_solution("task04_1")

## Part 3 - Application for a complex aquifer system

In [ ]:
du.display_image(
    image_filename='UnconfinedConfinedSandstoneAquiferDarcyUseExercise.png',
    image_folder='3_exercises', 
    )
check_task_with_solution("task_exercise_darcy_further_application_and_use_1")
check_task_with_solution("task_exercise_darcy_further_application_and_use_2")
check_task_with_solution("K_increase_sandstone_1")
check_task_with_solution("K_increase_sandstone_2")
check_task_with_solution("K_increase_sandstone_3")

What happens if the sandstone aquifer K value changes, 
if K larger, i.e. also larger transmissivity, what woukd happen?

- where would B shift (to A or C) if $h_A$ and $q_A$ value are fixed ---$h_A$ $q_A$ -> shift to A (see qc equation and if K increases, L-Lab must increase, Lab must decrease )
- if $q_A$ and B location are maintained, what would change to $h_A$ value --- $q_A$ B (see qc equation: qc increase because of K increase so Qc as well, then $q_A$ as well, and then if $q_A$ maintained: $h_A$ must increase)
- how would $q_A$ change (increase or decrease?) if we know for sure that B location does not change and $h_A$ neither --- $h_A$ B (see qc equation: qc increase because of K increase so Qc as well, then $q_A$ as well, and then if $h_A$ maintained: $q_A$ must increase)

In [12]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def run_simple_quiz():

    display(Markdown(r"""
## Task 3.3

Let us now look at the effects of increasing the Hydraulic Conductivity ($K$) of the sandstone aquifer.

We are interested in the effect it might have on both B location, $h_A$ or $q_A$.

For each question, we assume two of these three variables to be fixed and known, and wonder how the remaining one would change with an increase in $K$.
"""))

    # ----------------------------------
    # Question 1
    # ----------------------------------

    display(Markdown(r"""
### Question 1

If $h_A$ and $q_A$ are fixed, where would B shift?
"""))

    q1 = widgets.ToggleButtons(
        options=[
            "Shift towards A",
            "Shift towards C"
        ]
    )

    display(q1)

    # ----------------------------------
    # Question 2
    # ----------------------------------

    display(Markdown(r"""
### Question 2

If $q_A$ and B location are maintained, what happens to $h_A$?
"""))

    q2 = widgets.ToggleButtons(
        options=[
            "Increase",
            "Decrease"
        ]
    )

    display(q2)

    # ----------------------------------
    # Question 3
    # ----------------------------------

    display(Markdown(r"""
### Question 3

If B location and $h_A$ are fixed, what happens to $q_A$?
"""))

    q3 = widgets.ToggleButtons(
        options=[
            "Increase",
            "Decrease"
        ]
    )

    display(q3)

    # ----------------------------------
    # Buttons
    # ----------------------------------

    submit = widgets.Button(description="Submit")
    output = widgets.Output()

    def check_answers(b):

        with output:
            output.clear_output()

            if q1.value == "Shift towards A":
                display(Markdown("✅ Question 1 correct"))
            else:
                display(Markdown("❌ Question 1 incorrect"))

            if q2.value == "Increase":
                display(Markdown("✅ Question 2 correct"))
            else:
                display(Markdown("❌ Question 2 incorrect"))

            if q3.value == "Increase":
                display(Markdown("✅ Question 3 correct"))
            else:
                display(Markdown("❌ Question 3 incorrect"))

    submit.on_click(check_answers)

    display(submit)
    display(output)

run_simple_quiz()


## Task 3.3

Let us now look at the effects of increasing the Hydraulic Conductivity ($K$) of the sandstone aquifer.

We are interested in the effect it might have on both B location, $h_A$ or $q_A$.

For each question, we assume two of these three variables to be fixed and known, and wonder how the remaining one would change with an increase in $K$.



### Question 1

If $h_A$ and $q_A$ are fixed, where would B shift?


ToggleButtons(options=('Shift towards A', 'Shift towards C'), value='Shift towards A')


### Question 2

If $q_A$ and B location are maintained, what happens to $h_A$?


ToggleButtons(options=('Increase', 'Decrease'), value='Increase')


### Question 3

If B location and $h_A$ are fixed, what happens to $q_A$?


ToggleButtons(options=('Increase', 'Decrease'), value='Increase')

Button(description='Submit', style=ButtonStyle())

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

def run_simple_quiz():

    # ==========================================================
    # Intro
    # ==========================================================

    display(Markdown(r"""
## Task 3.3

Let us now look at the effects of increasing the Hydraulic Conductivity ($K$) of the sandstone aquifer.

We are interested in the effect it might have on both B location, $h_A$ or $q_A$.

For each question, we assume two of these three variables to be fixed and known, and wonder how the remaining one would change with an increase in $K$.
"""))

    # ==========================================================
    # Question 1
    # ==========================================================

    display(Markdown(r"""
### Question 1

If $h_A$ and $q_A$ are fixed, where would B shift?
"""))

    q1 = widgets.ToggleButtons(
        options=[
            "Shift towards A",
            "Shift towards C"
        ]
    )

    display(q1)

    # ==========================================================
    # Question 2
    # ==========================================================

    display(Markdown(r"""
### Question 2

If $q_A$ and B location are maintained, what happens to $h_A$?
"""))

    q2 = widgets.ToggleButtons(
        options=[
            "Increase",
            "Decrease"
        ]
    )

    display(q2)

    # ==========================================================
    # Question 3
    # ==========================================================

    display(Markdown(r"""
### Question 3

If B location and $h_A$ are fixed, how would $q_A$ change?
"""))

    q3 = widgets.ToggleButtons(
        options=[
            "Increase",
            "Decrease"
        ]
    )

    display(q3)

    # ==========================================================
    # Buttons
    # ==========================================================

    submit_btn = widgets.Button(
        description="Submit",
        button_style="primary"
    )

    solution_btn = widgets.Button(
        description="Show Solutions"
    )

    solution_btn.layout.display = "none"

    feedback_output = widgets.Output()
    solution_output = widgets.Output()

    # ==========================================================
    # Submit
    # ==========================================================

    def on_submit(b):

        with feedback_output:

            clear_output()

            score = 0

            if q1.value == "Shift towards A":
                display(Markdown("✅ **Question 1:** Correct"))
                score += 1
            else:
                display(Markdown("❌ **Question 1:** Incorrect"))

            if q2.value == "Increase":
                display(Markdown("✅ **Question 2:** Correct"))
                score += 1
            else:
                display(Markdown("❌ **Question 2:** Incorrect"))

            if q3.value == "Increase":
                display(Markdown("✅ **Question 3:** Correct"))
                score += 1
            else:
                display(Markdown("❌ **Question 3:** Incorrect"))

            display(Markdown(f"### Score: {score}/3"))

        solution_btn.layout.display = ""

    # ==========================================================
    # Solutions
    # ==========================================================

    def on_solution(b):

        with solution_output:

            clear_output()

            display(Markdown(r"""
### Question 1

**Correct answer:** Shift towards A

Since $K$ increases, transmissivity increases.

To maintain fixed $q_A$ with fixed $h_A$, the hydraulic gradient must decrease.

Therefore B shifts towards A, meaning $L_{ab}$ decreases.
"""))

            display(Markdown(r"""
### Question 2

**Correct answer:** Increase

If $K$ increases and geometry remains fixed, the system becomes more conductive.

To maintain the same flux $q_A$, the required driving head $h_A$ must increase.
"""))

            display(Markdown(r"""
### Question 3

**Correct answer:** Increase

With fixed geometry and fixed head $h_A$, Darcy's law implies that increasing $K$ increases the discharge.

Therefore $q_A$ increases.
"""))

    # ==========================================================
    # Connect callbacks
    # ==========================================================

    submit_btn.on_click(on_submit)
    solution_btn.on_click(on_solution)

    # ==========================================================
    # Display outputs
    # ==========================================================

    display(widgets.HBox([submit_btn, solution_btn]))
    display(feedback_output)
    display(solution_output)


run_simple_quiz()


## Task 3.3

Let us now look at the effects of increasing the Hydraulic Conductivity ($K$) of the sandstone aquifer.

We are interested in the effect it might have on both B location, $h_A$ or $q_A$.

For each question, we assume two of these three variables to be fixed and known, and wonder how the remaining one would change with an increase in $K$.



### Question 1

If $h_A$ and $q_A$ are fixed, where would B shift?


ToggleButtons(options=('Shift towards A', 'Shift towards C'), value='Shift towards A')


### Question 2

If $q_A$ and B location are maintained, what happens to $h_A$?


ToggleButtons(options=('Increase', 'Decrease'), value='Increase')


### Question 3

If B location and $h_A$ are fixed, how would $q_A$ change?


ToggleButtons(options=('Increase', 'Decrease'), value='Increase')

Output()

Output()